In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
adata = sc.read_h5ad("/data input")
print(adata)

output_dir = Path("FoldertoSaveFigures")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
obj = adata

#top 50 genes by log2 difference between the Stro-7 centroid and the highest other 
#fibroblast centroid (Stro-5, -8, -9, -10, -11), among genes detected in ≥1% of nuclei.

stro7_top50 = ['MGP', 'IGFBP5', 'CCDC80', 'GSN', 'ADH1B', 'ASPN', 'OGN', 'DCN', 'SERPINF1', 'HES1',
               'APOD', 'PRELP', 'SRPX', 'CTSK', 'SPARCL1', 'IGFBP7', 'CXCL12', 'MATN2', 'NDRG2', 'PLTP',
               'FBLN5', 'TIMP1', 'PLXDC2', 'RARRES2', 'MFAP4', 'GPNMB', 'CST3', 'SPOCK1', 'MFAP5', 'FXYD1',
               'OMD', 'PTGIS', 'HTRA1', 'PLAC9', 'AEBP1', 'PODN', 'SLC40A1', 'PDGFRL', 'PMP22', 'FBLN1',
               'TNXB', 'ACSM3', 'FAP', 'CCN5', 'CTSF', 'BGN', 'NFIA', 'A2M', 'LRP1', 'PID1']
modules_in = {"Stro7_top50": stro7_top50}

# ---------------- settings ----------------
group_key   = 'cell_subtype'
group_order = ['eFib1', 'eFib2', 'eFib3', 'eFib4', 'dec1', 'dec2', 'decAp1', 'decAp2']
group_map   = {g: g for g in group_order}   # edit to merge clusters, e.g. {'eFib1': 'eFib', ...}
min_pct     = 0.01     # gene must be detected in >= 1% of nuclei of at least one group
min_genes   = 4
out_prefix  = 'Ulrich2024_sg_'
vmin, vmax  = -1.8, 1.8
outpath = os.path.join(output_dir, "20261110-Human-Ulrich2024-Stro7top50-ModuleScore-heatmap.svg")

# ---------------- nuclei, log-normalised values in X ----------------
grp = obj.obs[group_key].astype(str).map(group_map)
cells = grp.notna().to_numpy()
group_order = [g for g in dict.fromkeys(group_map.values()) if g in set(grp[cells])]

tmp = ad.AnnData(
    X=obj.layers['log1p_counts'][cells],
    obs=pd.DataFrame({'group': pd.Categorical(grp[cells], categories=group_order)},
                     index=obj.obs_names[cells]),
    var=pd.DataFrame(index=obj.var_names))

# ---------------- C7 check ----------------
if 'C7' in tmp.var_names:
    c7 = tmp[:, 'C7'].X
    c7 = c7.toarray().ravel() if hasattr(c7, 'toarray') else np.asarray(c7).ravel()
    print("C7 detection per group (% of nuclei):")
    print((pd.Series(c7 > 0).groupby(tmp.obs['group'].to_numpy()).mean() * 100).round(2))
else:
    print("C7 is not in adata.var_names")

# ---------------- filter genes ----------------
all_genes = sorted({g for gs in modules_in.values() for g in gs})
present = [g for g in all_genes if g in tmp.var_names]
Xg = tmp[:, present].X
Xg = Xg.toarray() if hasattr(Xg, 'toarray') else np.asarray(Xg)
det = pd.DataFrame(Xg > 0, columns=present).groupby(tmp.obs['group'].to_numpy()).mean()
expressed = set(det.columns[(det >= min_pct).any(axis=0)])

modules = {m: [g for g in dict.fromkeys(gs) if g in expressed] for m, gs in modules_in.items()}
for m, gs in modules_in.items():
    dropped = [g for g in gs if g not in modules[m]]
    print(f"{m}: {len(modules[m])}/{len(gs)} genes kept; dropped: {dropped}")
scored = [m for m in modules if len(modules[m]) >= min_genes]

# ---------------- module score ----------------
for m in scored:
    sc.tl.score_genes(tmp, modules[m], score_name=f'sg_{m}', ctrl_size=50,
                      random_state=0, use_raw=False)
    obj.obs[f'{out_prefix}{m}'] = np.nan
    obj.obs.loc[tmp.obs_names, f'{out_prefix}{m}'] = tmp.obs[f'sg_{m}']

sg = tmp.obs[[f'sg_{m}' for m in scored]]
sg.columns = scored
mean_sg = sg.groupby(tmp.obs['group'], observed=True).mean().loc[group_order]
mat = (mean_sg - mean_sg.mean()) / mean_sg.std(ddof=0)        # z across groups, per module

print("\nMean module score per group:")
display(pd.concat({'mean_score': mean_sg, 'z': mat}, axis=1).round(3))

# ---------------- heatmap (groups as columns, one row per module) ----------------
plt.rcParams['svg.fonttype'] = 'none'
M = mat.T
fig, ax = plt.subplots(figsize=(0.8 * len(group_order) + 2, 0.6 * len(scored) + 1.8))
im = ax.imshow(M.values, cmap='viridis', vmin=vmin, vmax=vmax, aspect='auto')
ax.set_xticks(range(len(group_order)), group_order, rotation=45, ha='right')
ax.set_yticks(range(len(scored)), scored)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M.values[i, j]
        ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=9,
                color='black' if (v - vmin) / (vmax - vmin) > 0.6 else 'white')
ax.set_title('Stro-7 top-50 module score (z across groups)')
fig.colorbar(im, ax=ax, shrink=0.8, label='z')
plt.tight_layout()
fig.savefig(outpath, format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# Settings
key_sample   = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"

timepoints = ["secretory", "menstrual"]            # left to right
rotations  = {"secretory": -90, "menstrual": 90}   # degrees

score_col  = "Ulrich2024_sg_Stro7_top50"
cmap       = "viridis"
scale      = "shared"        # 'shared' (comparable across timepoints) or 'per_panel'
pct        = (1, 99)         # color range clipped to these percentiles
pad_frac   = 0.03            # margin around each section

show_unscored = True         # draw nuclei without a score underneath
other_color   = "#231f20"
dot_size      = 25           # point size of a 6-inch single image
fig_size_in   = 6
grid_panel_in = 3            # size of each panel (inches)
background    = "black"
dpi_save      = 300
outpath = os.path.join(output_dir, "20261110-Human-Ulrich2024-Stro7top50-spatial-grid.png")

# All nuclei of each section set the rotation center; falls back to adata if adata1 doesn't exist
full = adata1 if "adata1" in globals() else adata

# ---------------- checks ----------------
for name, a in [("adata", adata), ("full", full)]:
    for col in [key_sample, x_key, y_key]:
        if col not in a.obs:
            raise ValueError(f"{col} not found in {name}.obs")
    present = set(a.obs[key_sample].astype(str))
    missing_tp = [tp for tp in timepoints if tp not in present]
    if missing_tp:
        raise ValueError(f"No nuclei for {missing_tp} in {name}. Present: {sorted(present)}")
if score_col not in adata.obs:
    raise ValueError(f"{score_col} not in adata.obs; run the module-score cell first")

# ---------------- rotate each section around its own center ----------------
panels = {}
for tp in timepoints:
    fo = full.obs[full.obs[key_sample].astype(str) == tp]
    fx, fy = fo[x_key].astype(float).to_numpy(), fo[y_key].astype(float).to_numpy()
    ok = np.isfinite(fx) & np.isfinite(fy)
    cx, cy = 0.5 * (fx[ok].min() + fx[ok].max()), 0.5 * (fy[ok].min() + fy[ok].max())

    obs = adata.obs[adata.obs[key_sample].astype(str) == tp]
    x, y = obs[x_key].astype(float).to_numpy(), obs[y_key].astype(float).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, obs = x[ok], y[ok], obs.loc[ok]

    t = np.deg2rad(rotations[tp])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    panels[tp] = dict(xr=xr, yr=yr, s=obs[score_col].astype(float).to_numpy())

# Same physical scale in every panel: square window sized to the largest section
span = max(max(p["xr"].max() - p["xr"].min(), p["yr"].max() - p["yr"].min())
           for p in panels.values()) * (1 + 2 * pad_frac)
for p in panels.values():
    mx = 0.5 * (p["xr"].min() + p["xr"].max())
    my = 0.5 * (p["yr"].min() + p["yr"].max())
    p["xlim"] = (mx - span / 2, mx + span / 2)
    p["ylim"] = (my - span / 2, my + span / 2)

# ---------------- color ranges ----------------
if scale == "shared":
    allv = np.concatenate([p["s"] for p in panels.values()])
    lo, hi = np.nanpercentile(allv, pct)
    ranges = {tp: (lo, hi) for tp in timepoints}
else:
    ranges = {tp: tuple(np.nanpercentile(panels[tp]["s"], pct)) for tp in timepoints}

display(pd.DataFrame({
    "color min": [ranges[tp][0] for tp in timepoints],
    "color max": [ranges[tp][1] for tp in timepoints],
    "score min": [np.nanmin(panels[tp]["s"]) for tp in timepoints],
    "score max": [np.nanmax(panels[tp]["s"]) for tp in timepoints],
    "n scored nuclei": [int(np.isfinite(panels[tp]["s"]).sum()) for tp in timepoints],
}, index=pd.Index(timepoints, name=score_col))
    .style.format("{:.3f}").format("{:,}", subset=["n scored nuclei"]))

# ---------------- grid: panels touching, no labels ----------------
n = len(timepoints)
fig = plt.figure(figsize=(grid_panel_in * n, grid_panel_in))
fig.patch.set_facecolor(background)
dot_size_grid = dot_size * (grid_panel_in / fig_size_in) ** 2

for i, tp in enumerate(timepoints):
    p = panels[tp]
    ax = fig.add_axes([i / n, 0, 1 / n, 1])
    s = p["s"]
    has = np.isfinite(s)

    if show_unscored and (~has).any():
        ax.scatter(p["xr"][~has], p["yr"][~has], s=dot_size_grid, color=other_color,
                   linewidth=0, zorder=1)

    o = np.argsort(s[has])                       # highest scores drawn on top
    vmin, vmax = ranges[tp]
    ax.scatter(p["xr"][has][o], p["yr"][has][o], c=s[has][o], cmap=cmap,
               vmin=vmin, vmax=vmax, s=dot_size_grid, linewidth=0, zorder=2)

    ax.set_xlim(*p["xlim"])
    ax.set_ylim(*p["ylim"])
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
plt.show()
print(f"saved {outpath}")